# 03 – DQ Runner (schedule this)
1. Refresh the OU reference tables (customer → OU map, OU dimension)
2. For each active rule: count the population (`base_sql`) and the failing records
   (`failure_condition` on the base, or `failure_sql`) – **overall** and **per OU** – and store failing keys
3. Rebuild the current worklist and close the run. One failing rule does not stop the others.

Counts are **distinct record_key**. Overall counts go to `dq_rule_run_log`, per-OU counts to `dq_rule_run_log_ou`.
A customer with sites in several OUs is counted once overall and once in each of its OUs.

In [ ]:
%run 00_dq_config

In [ ]:
# Step 1: start the run
run_ts   = now_local()
run_date = run_ts.date()
run_id   = str(uuid.uuid4())

if REPLACE_SAME_DAY_RUN:
    for tbl in [T_RUN_LOG, T_RUN_LOG_OU, T_FAILED_HIST, T_RUN]:
        spark.sql(f"DELETE FROM {tbl} WHERE run_date = DATE'{run_date}'")

spark.sql(f'''
INSERT INTO {T_RUN} VALUES
('{run_id}', DATE'{run_date}', TIMESTAMP'{run_ts}', NULL, 'RUNNING', NULL, NULL)
''')
print(f"Run {run_id} started for {run_date}")

In [ ]:
# Step 2: refresh OU reference tables
# 2a. customer -> every OU where the account has a site
(spark.sql(CUSTOMER_OU_MAP_SQL)
      .select(F.col("cust_account_id").cast("string"), F.col("org_id").cast("int"))
      .where("org_id IS NOT NULL")
      .distinct()
      .write.mode("overwrite").format("delta").saveAsTable(T_CUST_OU_MAP))

# 2b. OU dimension = OU names + any org_id seen in the map + 'Unassigned', with region mapping
try:
    ou_src = spark.sql(OU_SOURCE_SQL).select(F.col("org_id").cast("int"), F.col("ou_name").cast("string"))
except Exception as e:
    print(f"OU source not available, using org_id as OU name: {str(e)[:150]}")
    ou_src = spark.createDataFrame([], "org_id INT, ou_name STRING")

map_ous  = spark.table(T_CUST_OU_MAP).select("org_id").distinct()
missing  = map_ous.join(ou_src, "org_id", "left_anti").withColumn("ou_name", F.col("org_id").cast("string"))
unassign = spark.createDataFrame([(-1, "Unassigned")], "org_id INT, ou_name STRING")

(ou_src.unionByName(missing).unionByName(unassign)
       .dropDuplicates(["org_id"])
       .join(spark.table(T_OU_REGION_MAP).dropDuplicates(["org_id"]), "org_id", "left")
       .select("org_id", "ou_name",
               F.coalesce("region",  F.lit("Unmapped")).alias("region"),
               F.coalesce("country", F.lit("Unmapped")).alias("country"))
       .write.mode("overwrite").format("delta").saveAsTable(T_DIM_OU))

In [ ]:
# Step 3: helper - give every row an org_id
def add_org_id(df, entity_grain):
    df = df.withColumn("record_key", F.col("record_key").cast("string"))
    if "org_id" in df.columns:                                   # rule returns its own OU (site-level rules)
        df = df.withColumn("org_id", F.col("org_id").cast("int"))
    elif entity_grain == "CUSTOMER":                             # customer rules: all OUs of the customer
        ou_map = spark.table(T_CUST_OU_MAP).withColumnRenamed("cust_account_id", "record_key")
        df = df.join(ou_map, "record_key", "left")
    else:
        df = df.withColumn("org_id", F.lit(None).cast("int"))
    return df.withColumn("org_id", F.coalesce(F.col("org_id"), F.lit(-1)))

In [ ]:
# Step 4: execute rules
rules = (spark.table(T_CATALOG)
              .filter("is_active = true")
              .orderBy("rule_id")
              .collect())

hist_cols = spark.table(T_FAILED_HIST).columns
log_rows, ou_rows = [], []

for r in rules:
    t0 = time.time()
    print(f"Rule {r.rule_id}: {r.rule_name}")
    base_o = None
    try:
        # population (denominator), overall and per OU
        base      = spark.sql(r.base_sql)
        base_cols = ["record_key"] + (["org_id"] if "org_id" in base.columns else [])
        base_o    = (add_org_id(base.select(*base_cols), r.entity_grain)
                         .select("record_key", "org_id").distinct().cache())
        total       = base_o.select("record_key").distinct().count()
        total_by_ou = {x.org_id: x.n for x in
                       base_o.groupBy("org_id").agg(F.countDistinct("record_key").alias("n")).collect()}

        # failing records -> history (with OU + optional record attributes)
        failed_src = base.filter(r.failure_condition) if r.failure_condition else spark.sql(r.failure_sql)
        keep       = ["record_key"] + [c for c in ["org_id"] + ATTRIBUTE_COLUMNS if c in failed_src.columns]
        failed_o   = add_org_id(failed_src.select(*keep), r.entity_grain)

        attrs = [F.col(c).cast("string").alias(c) if c in failed_o.columns
                 else F.lit(None).cast("string").alias(c) for c in ATTRIBUTE_COLUMNS]
        (failed_o.select(F.lit(run_id).alias("run_id"),
                         F.lit(run_date).cast("date").alias("run_date"),
                         F.lit(r.rule_id).cast("int").alias("rule_id"),
                         "record_key", "org_id", *attrs)
                 .distinct()
                 .select(*hist_cols)
                 .write.mode("append").format("delta").saveAsTable(T_FAILED_HIST))

        this_run     = (spark.table(T_FAILED_HIST)
                             .filter((F.col("run_date") == F.lit(run_date)) &
                                     (F.col("run_id") == run_id) &
                                     (F.col("rule_id") == r.rule_id)))
        failed       = this_run.select("record_key").distinct().count()
        failed_by_ou = {x.org_id: x.n for x in
                        this_run.groupBy("org_id").agg(F.countDistinct("record_key").alias("n")).collect()}

        # overall row
        passed    = total - failed
        pass_rate = round(passed / total, 6) if total > 0 else None
        log_rows.append((run_id, run_date, run_ts, r.rule_id, r.rule_version,
                         total, failed, passed, pass_rate, "SUCCESS", None,
                         round(time.time() - t0, 1)))

        # per-OU rows
        for ou in sorted(set(total_by_ou) | set(failed_by_ou)):
            t_ou = int(total_by_ou.get(ou, 0))
            f_ou = int(failed_by_ou.get(ou, 0))
            ou_rows.append((run_id, run_date, run_ts, r.rule_id, r.rule_version, int(ou),
                            t_ou, f_ou, t_ou - f_ou, round((t_ou - f_ou) / t_ou, 6) if t_ou > 0 else None))

        print(f"   total={total}  failed={failed}  pass_rate={pass_rate}  OUs={len(total_by_ou)}")

    except Exception as e:
        log_rows.append((run_id, run_date, run_ts, r.rule_id, r.rule_version,
                         None, None, None, None, "FAILED", str(e)[:2000],
                         round(time.time() - t0, 1)))
        print(f"   FAILED: {str(e)[:300]}")
    finally:
        if base_o is not None:
            base_o.unpersist()

In [ ]:
# Step 5: write the run logs (overall + per OU)
if log_rows:
    (spark.createDataFrame(log_rows, spark.table(T_RUN_LOG).schema)
          .write.mode("append").format("delta").saveAsTable(T_RUN_LOG))
if ou_rows:
    (spark.createDataFrame(ou_rows, spark.table(T_RUN_LOG_OU).schema)
          .write.mode("append").format("delta").saveAsTable(T_RUN_LOG_OU))

In [ ]:
# Step 6: rebuild the current worklist (failures of this run + how long each has been failing)
first_seen = (spark.table(T_FAILED_HIST)
                   .groupBy("rule_id", "record_key")
                   .agg(F.min("run_date").alias("first_failed_date")))

current = (spark.table(T_FAILED_HIST)
                .filter(F.col("run_id") == run_id)
                .join(first_seen, ["rule_id", "record_key"], "left")
                .withColumn("days_failing",
                            (F.datediff("run_date", "first_failed_date") + 1).cast("int")))

current.select(*spark.table(T_FAILED_CURR).columns) \
       .write.mode("overwrite").format("delta").saveAsTable(T_FAILED_CURR)

In [ ]:
# Step 7: history retention
spark.sql(f'''
DELETE FROM {T_FAILED_HIST}
WHERE run_date < date_sub(DATE'{run_date}', {HISTORY_RETENTION_DAYS})
''')

In [ ]:
# Step 8: close the run
n_exec  = len(log_rows)
n_error = sum(1 for x in log_rows if x[9] == "FAILED")
status  = "SUCCESS" if n_error == 0 else ("FAILED" if n_error == n_exec else "PARTIAL")

spark.sql(f'''
UPDATE {T_RUN}
SET end_ts = TIMESTAMP'{now_local()}',
    status = '{status}',
    rules_executed = {n_exec},
    rules_errored  = {n_error}
WHERE run_id = '{run_id}'
''')

print(f"Run {run_id} finished: {status} ({n_exec} rules, {n_error} errored)")
display(spark.table(T_RUN_LOG).filter(F.col("run_id") == run_id).orderBy("rule_id"))

In [ ]:
# Step 9: return status to the pipeline
notebookutils.notebook.exit(status)